
# 나만의 목소리 TTS 파인튜닝 (Google Colab)
이 노트북은 **Coqui TTS**로 기본 모델을 불러와 **내 목소리로 파인튜닝**하고,
결과 모델을 저장해 프로젝트에 넣는 흐름을 제공합니다.

**대상:** 한국어/다국어 TTS 파인튜닝 (VITS 기반)

---
## ✅ 준비물
- 10~60분 분량의 **내 목소리 음성 데이터** (더 많을수록 좋음)
- 각 음성에 대한 **정확한 텍스트** (문장 단위)

**데이터 구조 예시**
```
my_voice_dataset/
  wavs/
    0001.wav
    0002.wav
  metadata.csv
```
metadata.csv 형식 (파일명|텍스트)
```
0001.wav|안녕하세요. 테스트 문장입니다.
0002.wav|여기는 흡연 단속 시스템입니다.
```


In [ ]:

# @title 1) Colab 기본 세팅
!pip -q install TTS==0.22.0
!pip -q install librosa soundfile


In [ ]:

# @title 2) 구글 드라이브 마운트 (모델/데이터 저장용)
from google.colab import drive
drive.mount('/content/drive')

# 예시 경로 (필요에 맞게 변경)
DATA_DIR = '/content/drive/MyDrive/my_voice_dataset'
OUTPUT_DIR = '/content/drive/MyDrive/tts_finetune_output'



---
## 3) 데이터 체크 (샘플레이트 22.05kHz 권장)
- 필요 시 16k/22.05k/24k로 통일하세요.
- 잡음 제거/무음 구간 삭제를 권장합니다.


In [ ]:

# @title 데이터 존재 여부 확인
import os, glob
wav_files = glob.glob(os.path.join(DATA_DIR, 'wavs', '*.wav'))
print('WAV 파일 개수:', len(wav_files))
print('metadata.csv 존재?', os.path.exists(os.path.join(DATA_DIR, 'metadata.csv')))



---
## 4) 기본 모델 다운로드
- 한국어 기반 모델: `tts_models/ko/css10/vits`
- 다국어 기반 모델: `tts_models/multilingual/multi-dataset/your_tts`

원하는 기본 모델 이름으로 교체 가능합니다.


In [ ]:

# @title 기본 모델 가져오기
from TTS.utils.manage import ModelManager

manager = ModelManager()
BASE_MODEL = 'tts_models/ko/css10/vits'  # 필요 시 변경
model_path, config_path, _ = manager.download_model(BASE_MODEL)
print('모델 경로:', model_path)
print('설정 경로:', config_path)



---
## 5) 파인튜닝 설정 파일 만들기
- 기존 설정을 불러와 내 데이터셋 경로로 덮어씌웁니다.
- 학습 규모가 작을수록 learning rate를 낮추는 것이 안정적입니다.


In [ ]:

# @title 파인튜닝 설정 생성
import os
from TTS.config import load_config
from TTS.config.shared_configs import BaseDatasetConfig

config = load_config(config_path)
config.datasets = [
    BaseDatasetConfig(
        formatter='ljspeech',
        meta_file_train='metadata.csv',
        path=DATA_DIR,
    )
]
config.output_path = OUTPUT_DIR
config.run_name = 'my_voice_finetune'
config.batch_size = 16
config.eval_batch_size = 16
config.learning_rate = 1e-4
config.num_loader_workers = 2

finetune_config_path = '/content/finetune_config.json'
config.save_json(finetune_config_path)
print('저장됨:', finetune_config_path)



---
## 6) 파인튜닝 실행
- **GPU 런타임 권장** (Runtime → Change runtime type → GPU)
- 1~2시간 정도 소요될 수 있습니다.


In [ ]:

# @title 파인튜닝 시작
!python -m TTS.bin.train_tts   --config_path /content/finetune_config.json   --restore_path {model_path}



---
## 7) 학습된 모델로 음성 생성 테스트
- `best_model.pth` 생성 후 사용할 수 있습니다.
- 생성된 WAV를 다운로드하여 프로젝트에 넣으세요.


In [ ]:

# @title TTS 생성 테스트
from TTS.api import TTS

MODEL_DIR = f"{OUTPUT_DIR}/my_voice_finetune"
model_file = f"{MODEL_DIR}/best_model.pth"
config_file = f"{MODEL_DIR}/config.json"

tts = TTS(model_path=model_file, config_path=config_file, gpu=True)
tts.tts_to_file(text='거기 빨간 패딩 입은 사람, 금연 구역입니다. 담배 꺼!', file_path='sample.wav')
print('생성 완료: sample.wav')



---
## 8) 프로젝트에 넣는 방법 (예시)
로컬에서 실행할 때는 아래처럼 `TTS` 라이브러리로 생성한 WAV를 재생하면 됩니다.
```python
from TTS.api import TTS
tts = TTS(model_path='best_model.pth', config_path='config.json', gpu=False)
tts.tts_to_file(text='금연 구역입니다. 담배를 꺼주세요.', file_path='warning.wav')
# pygame/playsound 등으로 warning.wav 재생
```
※ 현재 코드의 ElevenLabs 부분을 이 방식으로 바꾸면 됩니다.
